# R Master v8b Lite · Head Beauty · 3-View Low Load

这是 v8b 的轻量修正版。只解决一个问题：

> **在 iPhone + Colab 场景下，尽量稳定地看清 Lap 头到底长什么样。**

### 改动
- 只渲染 3 张头肩图：正面 / 侧面 / 3/4；
- 640×640；
- **不再启动 Eevee 三点灯**；
- 改用 Workbench Texture 模式，优先稳定与真实 UV 贴图；
- 一次 Blender 进程连续出 3 张图，避免反复启动 Blender；
- 每张图生成后立刻写入 Drive；
- 如果中途断线，已有 PNG 仍保留；
- 复用 v8b 已生成的材质预览 .blend；如果不存在则先跑 v8b build cell 生成。

### 不做
- 不改 505 骨；
- 不转权重；
- 不焊 neck；
- 不烘焙 Rest Pose；
- 不导 VRM。


In [ ]:
from google.colab import drive, files
from pathlib import Path
import shutil, subprocess, zipfile, json, os

print("R Master v8b Lite · Head Beauty 3-View")
drive.mount("/content/drive")

ROOT=Path("/content/drive/MyDrive/R_Master")
SRC=ROOT/"v8b_head_beauty"/"latest"/"R_Master_v8b_HEAD_BEAUTY_PREVIEW.blend"
CACHE=ROOT/"cache"
OUT=ROOT/"v8b_head_beauty"/"lite_latest"
OUT.mkdir(parents=True,exist_ok=True)

if not SRC.exists() or SRC.stat().st_size<50*1024*1024:
    raise RuntimeError("没有找到 v8b beauty preview .blend。先运行原 v8b 到 build 完成即可，不需要进入渲染。")
print(f"✓ v8b source：{SRC.stat().st_size/1024/1024:.1f} MiB")



In [ ]:
BLENDER_VERSION="4.4.3"
BLENDER_URL="https://download.blender.org/release/Blender4.4/blender-4.4.3-linux-x64.tar.xz"
LOCAL=Path("/content/r_master_v8b_lite"); LOCAL.mkdir(parents=True,exist_ok=True)
ARCHIVE=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
BDIR=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64"
DRIVE_ARCHIVE=CACHE/ARCHIVE.name

if shutil.which("xvfb-run") is None:
    subprocess.run(["apt-get","update","-qq"],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)
    subprocess.run(["apt-get","install","-y","-qq","xvfb","libgl1","libx11-6","libxi6","libxrender1","libxfixes3","libxkbcommon0","libsm6"],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)

if DRIVE_ARCHIVE.exists() and DRIVE_ARCHIVE.stat().st_size>100*1024*1024:
    shutil.copy2(DRIVE_ARCHIVE,ARCHIVE)
    print("✓ 复用 Blender 缓存")
else:
    subprocess.run(["wget","-q","--show-progress","-O",str(ARCHIVE),BLENDER_URL],check=True)
    shutil.copy2(ARCHIVE,DRIVE_ARCHIVE)

if not (BDIR/"blender").exists():
    if BDIR.exists(): shutil.rmtree(BDIR)
    subprocess.run(["tar","-xf",str(ARCHIVE),"-C",str(LOCAL)],check=True)

BLENDER=BDIR/"blender"
print("✓ Blender ready")



In [ ]:
RENDER=LOCAL/"R_Master_v8b_Lite_Render.py"
RENDER.write_text("\nimport bpy, os, sys\nfrom mathutils import Vector\n\nargv=sys.argv[sys.argv.index(\"--\")+1:] if \"--\" in sys.argv else []\nout=None\nfor i,a in enumerate(argv):\n    if a==\"--out\" and i+1<len(argv): out=argv[i+1]\nif not out: raise RuntimeError(\"missing --out\")\nos.makedirs(out,exist_ok=True)\n\nbody=bpy.data.objects.get(\"R2_Mona_Main\")\nrig=bpy.data.objects.get(\"R_Master_Align_v2_PREVIEW\") or bpy.data.objects.get(\"Mona_Armature\")\ndonors=[o for o in bpy.data.objects if o.type==\"MESH\" and o.get(\"R_DONOR\")==\"LapineHead_v8a_fix2\"]\nif not body or not rig or not donors:\n    raise RuntimeError(\"v8b preview objects missing\")\n\n# 只显示身体 + Lap 头 donor。\nfor o in bpy.context.scene.objects:\n    if o.type==\"ARMATURE\":\n        o.hide_render=True\n    if o.type==\"MESH\":\n        o.hide_render=(o!=body and o not in donors)\n\nscene=bpy.context.scene\nscene.render.engine=\"BLENDER_WORKBENCH\"\nscene.render.image_settings.file_format=\"PNG\"\nscene.render.resolution_x=640\nscene.render.resolution_y=640\nscene.render.resolution_percentage=100\nscene.display.shading.light=\"STUDIO\"\nscene.display.shading.color_type=\"TEXTURE\"\nscene.display.shading.show_shadows=True\nscene.display.shading.show_cavity=True\nscene.display.shading.cavity_type=\"WORLD\"\nscene.display.shading.show_specular_highlight=True\nscene.display.shading.background_type=\"VIEWPORT\"\nscene.display.shading.background_color=(0.035,0.035,0.045)\n\n# 用 Mona body bbox 决定尺度，避免 donor 异常导致相机飞。\nbp=[body.matrix_world@Vector(c) for c in body.bound_box]\nmn=Vector((min(p.x for p in bp),min(p.y for p in bp),min(p.z for p in bp)))\nmx=Vector((max(p.x for p in bp),max(p.y for p in bp),max(p.z for p in bp)))\nbc=(mn+mx)*0.5\nh=mx.z-mn.z\nd=h*2.7\n\nhb=rig.data.bones.get(\"Head\")\nHC=rig.matrix_world@hb.head_local if hb else Vector((bc.x,bc.y,mx.z-h*0.12))\ntarget=HC+Vector((0,0,h*0.075))\n\n# 清旧 camera，建一个新 camera。\nfor o in list(bpy.data.objects):\n    if o.type==\"CAMERA\":\n        bpy.data.objects.remove(o,do_unlink=True)\ncd=bpy.data.cameras.new(\"R_v8b_lite_cam_data\")\ncam=bpy.data.objects.new(\"R_v8b_lite_cam\",cd)\nscene.collection.objects.link(cam)\nscene.camera=cam\ncam.data.type=\"ORTHO\"\ncam.data.ortho_scale=h*0.30\n\ndef look_at(t):\n    cam.rotation_euler=(Vector(t)-cam.location).to_track_quat(\"-Z\",\"Y\").to_euler()\n\nviews=[\n    (\"head_front\",\"R_Master_v8b_Lite_head_front.png\",(HC.x,HC.y-d,target.z)),\n    (\"head_side\",\"R_Master_v8b_Lite_head_side.png\",(HC.x+d,HC.y,target.z)),\n    (\"head_three_quarter\",\"R_Master_v8b_Lite_head_three_quarter.png\",(HC.x+d*0.72,HC.y-d*0.72,target.z)),\n]\n\nfor idx,(name,filename,pos) in enumerate(views,1):\n    p=os.path.join(out,filename)\n    if os.path.exists(p) and os.path.getsize(p)>15000:\n        print(f\"[v8b Lite] CHECKPOINT {idx}/3 {name}\")\n        continue\n    cam.location=Vector(pos)\n    look_at(target)\n    scene.render.filepath=p\n    bpy.ops.render.render(write_still=True)\n    print(f\"[v8b Lite] RENDER_OK {idx}/3 {name}\",flush=True)\n\nprint(\"[v8b Lite] ALL_DONE\")\n",encoding="utf-8")

names=[
"R_Master_v8b_Lite_head_front.png",
"R_Master_v8b_Lite_head_side.png",
"R_Master_v8b_Lite_head_three_quarter.png"
]
if all((OUT/n).exists() and (OUT/n).stat().st_size>15000 for n in names):
    print("✓ 3 张头肩图已存在，跳过渲染")
else:
    log=OUT/"R_Master_v8b_Lite_render.log"
    cmd=["xvfb-run","-a",str(BLENDER),"--background",str(SRC),"--python",str(RENDER),"--","--out",str(OUT)]
    with log.open("w",encoding="utf-8") as f:
        p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout:
            f.write(line)
            if "[v8b Lite]" in line or "Traceback" in line or "RuntimeError" in line:
                print(line.rstrip(),flush=True)
        rc=p.wait()
    if rc!=0:
        print(log.read_text(encoding="utf-8",errors="replace")[-12000:])
        raise RuntimeError("v8b Lite 渲染失败")
print("✓ v8b Lite complete")



In [ ]:
from IPython.display import display, Image, Markdown
items=[
("头肩正面","R_Master_v8b_Lite_head_front.png"),
("头肩侧面","R_Master_v8b_Lite_head_side.png"),
("头肩 3/4","R_Master_v8b_Lite_head_three_quarter.png")
]
for title,f in items:
    display(Markdown("### "+title))
    display(Image(filename=str(OUT/f),width=520))

z=OUT/"R_Master_v8b_Lite_Review.zip"
if z.exists(): z.unlink()
with zipfile.ZipFile(z,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=6) as w:
    for _,f in items:
        w.write(OUT/f,arcname=f)
    log=OUT/"R_Master_v8b_Lite_render.log"
    if log.exists():
        w.write(log,arcname=log.name)

print(f"✓ Review ZIP：{z.stat().st_size/1024/1024:.1f} MiB")
files.download(str(z))

